In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cell 2 — Load data
df = pd.read_excel('marketing_campaign.xlsx')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

Shape: (2240, 29)
Columns: ['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']


,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,AcceptedCmp3,AcceptedCmp4,AcceptedCmp5,AcceptedCmp1,AcceptedCmp2,Complain,Z_CostContact,Z_Revenue,Response
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,7,0,0,0,0,0,0,3,11,1
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,5,0,0,0,0,0,0,3,11,0
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,4,0,0,0,0,0,0,3,11,0
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,6,0,0,0,0,0,0,3,11,0
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,5,0,0,0,0,0,0,3,11,0


In [2]:
# Cell 3 — Data Cleaning & Load into SQLite

# Create age column
df['Age'] = 2024 - df['Year_Birth']

# Create total spend column
df['TotalSpend'] = (df['MntWines'] + df['MntFruits'] + 
                    df['MntMeatProducts'] + df['MntFishProducts'] + 
                    df['MntSweetProducts'] + df['MntGoldProds'])

# Create total purchases column
df['TotalPurchases'] = (df['NumWebPurchases'] + df['NumCatalogPurchases'] + 
                        df['NumStorePurchases'] + df['NumDealsPurchases'])

# Create total campaigns accepted
df['TotalCampaignsAccepted'] = (df['AcceptedCmp1'] + df['AcceptedCmp2'] + 
                                 df['AcceptedCmp3'] + df['AcceptedCmp4'] + 
                                 df['AcceptedCmp5'] + df['Response'])

# Create age groups
df['AgeGroup'] = pd.cut(df['Age'], 
                         bins=[0, 30, 40, 50, 60, 100],
                         labels=['<30', '30-40', '40-50', '50-60', '60+'])

# Create income groups
df['IncomeGroup'] = pd.cut(df['Income'],
                            bins=[0, 30000, 50000, 70000, 100000, 999999],
                            labels=['Low', 'Lower-Mid', 'Mid', 'Upper-Mid', 'High'])

# Drop rows with missing income
df = df.dropna(subset=['Income'])

print("Cleaned shape:", df.shape)
print("\nNew columns added:")
print(df[['Age', 'TotalSpend', 'TotalPurchases', 
          'TotalCampaignsAccepted', 'AgeGroup', 'IncomeGroup']].head())

# Load into SQLite
conn = sqlite3.connect('marketing_campaign.db')
df.to_sql('marketing', conn, if_exists='replace', index=False)
print("\n✅ Data loaded into SQLite!")
print("Total records:", pd.read_sql_query(
    "SELECT COUNT(*) as total FROM marketing", conn).values[0][0])

Cleaned shape: (2216, 35)

New columns added:
   Age  TotalSpend  TotalPurchases  TotalCampaignsAccepted AgeGroup  \
0   67        1617              25                       1      60+   
1   70          27               6                       0      60+   
2   59         776              21                       0    50-60   
3   40          53               8                       0    30-40   
4   43         422              19                       0    40-50   

  IncomeGroup  
0         Mid  
1   Lower-Mid  
2   Upper-Mid  
3         Low  
4         Mid  

✅ Data loaded into SQLite!
Total records: 2216


In [3]:
 # Cell 4 — SQL Query 1: Overall Campaign Metrics
q1 = pd.read_sql_query("""
    SELECT 
        COUNT(*) AS total_customers,
        ROUND(AVG(Income), 2) AS avg_income,
        ROUND(AVG(TotalSpend), 2) AS avg_spend,
        ROUND(AVG(TotalPurchases), 2) AS avg_purchases,
        SUM(Response) AS total_conversions,
        ROUND(AVG(Response) * 100, 2) AS conversion_rate,
        ROUND(AVG(TotalCampaignsAccepted), 2) AS avg_campaigns_accepted
    FROM marketing
""", conn)
print("=== OVERALL CAMPAIGN METRICS ===")
print(q1.to_string(index=False))

=== OVERALL CAMPAIGN METRICS ===
 total_customers  avg_income  avg_spend  avg_purchases  total_conversions  conversion_rate  avg_campaigns_accepted
            2216    52247.25     607.08          14.88                333            15.03                    0.45


In [4]:
# Cell 5 — SQL Query 2: Channel Performance Analysis
q2 = pd.read_sql_query("""
    SELECT 
        'Web' AS channel,
        SUM(NumWebPurchases) AS total_purchases,
        ROUND(AVG(NumWebPurchases), 2) AS avg_purchases_per_customer,
        ROUND(SUM(NumWebPurchases) * 100.0 / SUM(TotalPurchases), 2) AS channel_share_pct
    FROM marketing
    UNION ALL
    SELECT 
        'Store' AS channel,
        SUM(NumStorePurchases),
        ROUND(AVG(NumStorePurchases), 2),
        ROUND(SUM(NumStorePurchases) * 100.0 / SUM(TotalPurchases), 2)
    FROM marketing
    UNION ALL
    SELECT 
        'Catalogue' AS channel,
        SUM(NumCatalogPurchases),
        ROUND(AVG(NumCatalogPurchases), 2),
        ROUND(SUM(NumCatalogPurchases) * 100.0 / SUM(TotalPurchases), 2)
    FROM marketing
    UNION ALL
    SELECT 
        'Deals' AS channel,
        SUM(NumDealsPurchases),
        ROUND(AVG(NumDealsPurchases), 2),
        ROUND(SUM(NumDealsPurchases) * 100.0 / SUM(TotalPurchases), 2)
    FROM marketing
    ORDER BY total_purchases DESC
""", conn)
print("=== CHANNEL PERFORMANCE ===")
print(q2.to_string(index=False))

=== CHANNEL PERFORMANCE ===
  channel  total_purchases  avg_purchases_per_customer  channel_share_pct
    Store            12855                        5.80              38.98
      Web             9053                        4.09              27.45
Catalogue             5919                        2.67              17.95
    Deals             5149                        2.32              15.61


In [5]:
# Cell 6 — SQL Query 3: Product Revenue Analysis
q3 = pd.read_sql_query("""
    SELECT
        'Wines' AS product, SUM(MntWines) AS total_revenue,
        ROUND(AVG(MntWines), 2) AS avg_per_customer,
        ROUND(SUM(MntWines) * 100.0 / SUM(TotalSpend), 2) AS revenue_share_pct
    FROM marketing
    UNION ALL
    SELECT 'Meat', SUM(MntMeatProducts), 
        ROUND(AVG(MntMeatProducts), 2),
        ROUND(SUM(MntMeatProducts) * 100.0 / SUM(TotalSpend), 2)
    FROM marketing
    UNION ALL
    SELECT 'Gold', SUM(MntGoldProds),
        ROUND(AVG(MntGoldProds), 2),
        ROUND(SUM(MntGoldProds) * 100.0 / SUM(TotalSpend), 2)
    FROM marketing
    UNION ALL
    SELECT 'Fish', SUM(MntFishProducts),
        ROUND(AVG(MntFishProducts), 2),
        ROUND(SUM(MntFishProducts) * 100.0 / SUM(TotalSpend), 2)
    FROM marketing
    UNION ALL
    SELECT 'Sweets', SUM(MntSweetProducts),
        ROUND(AVG(MntSweetProducts), 2),
        ROUND(SUM(MntSweetProducts) * 100.0 / SUM(TotalSpend), 2)
    FROM marketing
    UNION ALL
    SELECT 'Fruits', SUM(MntFruits),
        ROUND(AVG(MntFruits), 2),
        ROUND(SUM(MntFruits) * 100.0 / SUM(TotalSpend), 2)
    FROM marketing
    ORDER BY total_revenue DESC
""", conn)
print("=== PRODUCT REVENUE ANALYSIS ===")
print(q3.to_string(index=False))

=== PRODUCT REVENUE ANALYSIS ===
product  total_revenue  avg_per_customer  revenue_share_pct
  Wines         676083            305.09              50.26
   Meat         370063            167.00              27.51
   Gold          97427             43.97               7.24
   Fish          83405             37.64               6.20
 Sweets          59896             27.03               4.45
 Fruits          58405             26.36               4.34


In [6]:
# Cell 7 — SQL Query 4: Customer Segment Analysis
q4 = pd.read_sql_query("""
    SELECT
        Education,
        COUNT(*) AS total_customers,
        ROUND(AVG(Income), 2) AS avg_income,
        ROUND(AVG(TotalSpend), 2) AS avg_spend,
        ROUND(AVG(Response) * 100, 2) AS conversion_rate,
        ROUND(AVG(TotalCampaignsAccepted), 2) AS avg_campaigns_accepted
    FROM marketing
    GROUP BY Education
    ORDER BY avg_spend DESC
""", conn)
print("=== CUSTOMER SEGMENT BY EDUCATION ===")
print(q4.to_string(index=False))

=== CUSTOMER SEGMENT BY EDUCATION ===
 Education  total_customers  avg_income  avg_spend  conversion_rate  avg_campaigns_accepted
       PhD              481    56145.31     676.73            21.00                    0.55
Graduation             1116    52720.37     621.69            13.62                    0.44
    Master              365    52917.53     609.77            15.34                    0.43
  2n Cycle              200    47633.19     494.93            11.00                    0.36
     Basic               54    20306.26      81.80             3.70                    0.15


In [7]:
# Cell 8 — SQL Query 5: Income Group vs Spend & Conversion
q5 = pd.read_sql_query("""
    SELECT
        IncomeGroup,
        COUNT(*) AS total_customers,
        ROUND(AVG(Income), 2) AS avg_income,
        ROUND(AVG(TotalSpend), 2) AS avg_spend,
        ROUND(AVG(TotalPurchases), 2) AS avg_purchases,
        ROUND(AVG(Response) * 100, 2) AS conversion_rate,
        SUM(Response) AS total_conversions
    FROM marketing
    WHERE IncomeGroup IS NOT NULL
    GROUP BY IncomeGroup
    ORDER BY avg_spend DESC
""", conn)
print("=== INCOME GROUP ANALYSIS ===")
print(q5.to_string(index=False))

=== INCOME GROUP ANALYSIS ===
IncomeGroup  total_customers  avg_income  avg_spend  avg_purchases  conversion_rate  total_conversions
  Upper-Mid              495    79102.75    1391.09          20.79            27.88                138
       High               13   176835.62     829.62          20.15            30.77                  4
        Mid              648    60108.56     754.60          19.41            10.34                 67
  Lower-Mid              690    39751.13     188.72          10.32            12.32                 85
        Low              370    21477.18      72.18           7.36            10.54                 39


In [8]:
# Cell 9 — Export all data for Power BI
import os
os.makedirs('powerbi_exports', exist_ok=True)

# Export 1 — Main cleaned dataset
df.to_csv('powerbi_exports/marketing_clean.csv', index=False)
print("✅ marketing_clean.csv")

# Export 2 — Channel performance
q2.to_csv('powerbi_exports/channel_performance.csv', index=False)
print("✅ channel_performance.csv")

# Export 3 — Product revenue
q3.to_csv('powerbi_exports/product_revenue.csv', index=False)
print("✅ product_revenue.csv")

# Export 4 — Education segments
q4.to_csv('powerbi_exports/education_segments.csv', index=False)
print("✅ education_segments.csv")

# Export 5 — Income groups
q5.to_csv('powerbi_exports/income_groups.csv', index=False)
print("✅ income_groups.csv")

# Export 6 — Age group analysis
age_analysis = df.groupby('AgeGroup', observed=True).agg(
    total_customers=('ID', 'count'),
    avg_spend=('TotalSpend', 'mean'),
    avg_income=('Income', 'mean'),
    conversion_rate=('Response', lambda x: round(x.mean()*100, 2)),
    total_conversions=('Response', 'sum')
).round(2).reset_index()
age_analysis.to_csv('powerbi_exports/age_analysis.csv', index=False)
print("✅ age_analysis.csv")

print("\n🎉 All 6 files exported to powerbi_exports folder!")
print("📁 Location:", os.path.abspath('powerbi_exports'))

✅ marketing_clean.csv
✅ channel_performance.csv
✅ product_revenue.csv
✅ education_segments.csv
✅ income_groups.csv
✅ age_analysis.csv

🎉 All 6 files exported to powerbi_exports folder!
📁 Location: C:\Users\Life\OneDrive\Desktop\ML Project\multi market campaign\powerbi_exports
